In [1]:
import pandas as pd
import json
import os
from tqdm import tqdm
from collections import Counter, defaultdict
import pickle
import csv
import argparse

In [ ]:
df_raw = pd.read_csv('../2_2_clustering/data_mideast/cluster/time_t1.0_n50_m20/t1_m20_raw.csv', sep='\t', keep_default_na=False)

def combine_columns(row):
    if row['Event Mode'] == '':
        return row['Event Type']
    else:
        return str(row['Event Type']) + "_" + str(row['Event Mode'])

# 创建新列 'c'，应用上面定义的函数
df_raw['Relation'] = df_raw.apply(combine_columns, axis=1)

In [ ]:
df_raw

,Event ID,Actor Name,Event Type,Event Mode,Recipient Name,Event Date,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,Topic,Nday,Relation
0,20180101-0491-8557544d1585_ASSAULT_abduct,Essam Awad,ASSAULT,abduct,Al-Azhar Al-Sharif,2018-01-01,,Egypt,Egypt,,Barakat | Suleiman Al - Atifi | Essam Barakat ...,the House of Representatives,,20180101-0491-8557544d1585,-1,0,ASSAULT_abduct
1,20180101-0940-40bccf948bda_AGREE,Ans tribe,AGREE,,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,AGREE
2,20180101-0940-40bccf948bda_CONCEDE,Ans tribe,CONCEDE,,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,CONCEDE
3,20180101-0940-40bccf948bda_ACCUSE_disapprove,Ans tribe,ACCUSE,disapprove,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,ACCUSE_disapprove
4,20180101-0940-40bccf948bda_THREATEN,Ans tribe,THREATEN,,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,THREATEN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8984968,20240430-4226-c46141aefac2_CONSULT,Hama,CONSULT,,Hama,2024-04-30,diplomatic,Syria; Palestinian Territories,Syria; Palestinian Territories,CHN,Lin Jian,Fatah | Foreign Ministry | Hamas | the Chinese...,Palestine | Peopleâs Republic of China | Bei...,20240430-4226-c46141aefac2,3,2311,CONSULT
8984969,20240430-5267-f39bcb7d805b_CONSULT,Rania Al-Mashat,CONSULT,,Abdel Fattah el-Sisi,2024-04-30,economic | inequality,Egypt,Egypt,KEN,Abdel Fattah Al - Sisi | Rania Al - Mashat,IDA | the World Bank âs,Arab Republic of Egypt | Republic of Kenya | A...,20240430-5267-f39bcb7d805b,-1,2311,CONSULT
8984970,20240430-5267-f39bcb7d805b_AID,Rania Al-Mashat,AID,,Abdel Fattah el-Sisi,2024-04-30,economic | inequality,Egypt,Egypt,KEN,Abdel Fattah Al - Sisi | Rania Al - Mashat,IDA | the World Bank âs,Arab Republic of Egypt | Republic of Kenya | A...,20240430-5267-f39bcb7d805b,-1,2311,AID
8984971,20240430-5543-151220515edd_CONSULT,Bidzina Ivanishvili,CONSULT,,Party,2024-04-30,pro_democracy,Georgia,United States; None,GEO,Saakashvili | Bidzina Ivanishvili | Ivanishvil...,Georgian Dream,European Union | West Reef | Georgia | Tbilisi...,20240430-5543-151220515edd,-1,2311,CONSULT


In [4]:
md52rawtopic = {}
for rowid, row in tqdm(df_raw.iterrows(), total=len(df_raw)):
    md52rawtopic[row['Md5']] = row['Topic']

100%|██████████| 8984973/8984973 [04:21<00:00, 34316.35it/s]


In [ ]:
json.dump(md52rawtopic, open('./data_mideast_max100_min20/md52rawtopic.json', 'w'), indent=4)

In [6]:
df_raw = df_raw.rename(columns={'Actor Name': 'Actor1Name', 'Recipient Name': 'Actor2Name', 'Relation': 'EventType', 'Event Date': 'day', 'Nday': 'timid'})

In [7]:
df_raw

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,Topic,timid,EventType
0,20180101-0491-8557544d1585_ASSAULT_abduct,Essam Awad,ASSAULT,abduct,Al-Azhar Al-Sharif,2018-01-01,,Egypt,Egypt,,Barakat | Suleiman Al - Atifi | Essam Barakat ...,the House of Representatives,,20180101-0491-8557544d1585,-1,0,ASSAULT_abduct
1,20180101-0940-40bccf948bda_AGREE,Ans tribe,AGREE,,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,AGREE
2,20180101-0940-40bccf948bda_CONCEDE,Ans tribe,CONCEDE,,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,CONCEDE
3,20180101-0940-40bccf948bda_ACCUSE_disapprove,Ans tribe,ACCUSE,disapprove,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,ACCUSE_disapprove
4,20180101-0940-40bccf948bda_THREATEN,Ans tribe,THREATEN,,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,THREATEN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8984968,20240430-4226-c46141aefac2_CONSULT,Hama,CONSULT,,Hama,2024-04-30,diplomatic,Syria; Palestinian Territories,Syria; Palestinian Territories,CHN,Lin Jian,Fatah | Foreign Ministry | Hamas | the Chinese...,Palestine | Peopleâs Republic of China | Bei...,20240430-4226-c46141aefac2,3,2311,CONSULT
8984969,20240430-5267-f39bcb7d805b_CONSULT,Rania Al-Mashat,CONSULT,,Abdel Fattah el-Sisi,2024-04-30,economic | inequality,Egypt,Egypt,KEN,Abdel Fattah Al - Sisi | Rania Al - Mashat,IDA | the World Bank âs,Arab Republic of Egypt | Republic of Kenya | A...,20240430-5267-f39bcb7d805b,-1,2311,CONSULT
8984970,20240430-5267-f39bcb7d805b_AID,Rania Al-Mashat,AID,,Abdel Fattah el-Sisi,2024-04-30,economic | inequality,Egypt,Egypt,KEN,Abdel Fattah Al - Sisi | Rania Al - Mashat,IDA | the World Bank âs,Arab Republic of Egypt | Republic of Kenya | A...,20240430-5267-f39bcb7d805b,-1,2311,AID
8984971,20240430-5543-151220515edd_CONSULT,Bidzina Ivanishvili,CONSULT,,Party,2024-04-30,pro_democracy,Georgia,United States; None,GEO,Saakashvili | Bidzina Ivanishvili | Ivanishvil...,Georgian Dream,European Union | West Reef | Georgia | Tbilisi...,20240430-5543-151220515edd,-1,2311,CONSULT


In [8]:
grouplist = list(df_raw.groupby(['Actor1Name', 'EventType', 'Actor2Name', 'day', 'Topic', 'timid']))

In [9]:
grouplist[0][1]

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,Topic,timid,EventType
2945359,20190516-4135-a7cc3fab9ae4_AID,,AID,,Japan,2019-05-16,legislative | diplomatic,None; None,Japan,CAN,Shinzo | ( Jean - Claude ) Juncker | Kenji Yam...,Tokyo Electric Power Company Holdings Inc. 's ...,Brussels | Fukushima-ken | Europe | Japan,20190516-4135-a7cc3fab9ae4,-1,500,AID


In [10]:
grouplist[0][0]

('', 'AID', 'Japan', '2019-05-16', np.int64(-1), np.int64(500))

In [11]:
ce_pds = []
same_num = 0
num = 0
for group in grouplist:
    group_df = group[1]
    md5_list = list(group_df['Md5'].unique())
    if len(md5_list)>1:
        print("###")
        print(group_df)
        print("###")
        same_num = same_num + len(md5_list)
        num = num + 1
    ce_pd = group_df.head(1).copy()
    ce_pd['Md5_list'] = ', '.join(md5_list)
    ce_pds.append(ce_pd.copy())

In [12]:
md5_list

['20180512-0829-1d726338046b']

In [13]:
same_num

0

In [14]:
num

0

In [15]:
ce_df = pd.concat(ce_pds).sort_index()

In [16]:
ce_df

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,Topic,timid,EventType,Md5_list
0,20180101-0491-8557544d1585_ASSAULT_abduct,Essam Awad,ASSAULT,abduct,Al-Azhar Al-Sharif,2018-01-01,,Egypt,Egypt,,Barakat | Suleiman Al - Atifi | Essam Barakat ...,the House of Representatives,,20180101-0491-8557544d1585,-1,0,ASSAULT_abduct,20180101-0491-8557544d1585
1,20180101-0940-40bccf948bda_AGREE,Ans tribe,AGREE,,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,AGREE,20180101-0940-40bccf948bda
2,20180101-0940-40bccf948bda_CONCEDE,Ans tribe,CONCEDE,,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,CONCEDE,20180101-0940-40bccf948bda
3,20180101-0940-40bccf948bda_ACCUSE_disapprove,Ans tribe,ACCUSE,disapprove,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,ACCUSE_disapprove,20180101-0940-40bccf948bda
4,20180101-0940-40bccf948bda_THREATEN,Ans tribe,THREATEN,,Houthi movement,2018-01-01,military | human_rights,,Yemen,IRN,Ali Abdullah Saleh,Al - Aamas,Zamar | Al Wakrah | Sanaa | á¸¨aÅ£Å£Ät | HamdÄn,20180101-0940-40bccf948bda,-1,0,THREATEN,20180101-0940-40bccf948bda
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8984968,20240430-4226-c46141aefac2_CONSULT,Hama,CONSULT,,Hama,2024-04-30,diplomatic,Syria; Palestinian Territories,Syria; Palestinian Territories,CHN,Lin Jian,Fatah | Foreign Ministry | Hamas | the Chinese...,Palestine | Peopleâs Republic of China | Bei...,20240430-4226-c46141aefac2,3,2311,CONSULT,20240430-4226-c46141aefac2
8984969,20240430-5267-f39bcb7d805b_CONSULT,Rania Al-Mashat,CONSULT,,Abdel Fattah el-Sisi,2024-04-30,economic | inequality,Egypt,Egypt,KEN,Abdel Fattah Al - Sisi | Rania Al - Mashat,IDA | the World Bank âs,Arab Republic of Egypt | Republic of Kenya | A...,20240430-5267-f39bcb7d805b,-1,2311,CONSULT,20240430-5267-f39bcb7d805b
8984970,20240430-5267-f39bcb7d805b_AID,Rania Al-Mashat,AID,,Abdel Fattah el-Sisi,2024-04-30,economic | inequality,Egypt,Egypt,KEN,Abdel Fattah Al - Sisi | Rania Al - Mashat,IDA | the World Bank âs,Arab Republic of Egypt | Republic of Kenya | A...,20240430-5267-f39bcb7d805b,-1,2311,AID,20240430-5267-f39bcb7d805b
8984971,20240430-5543-151220515edd_CONSULT,Bidzina Ivanishvili,CONSULT,,Party,2024-04-30,pro_democracy,Georgia,United States; None,GEO,Saakashvili | Bidzina Ivanishvili | Ivanishvil...,Georgian Dream,European Union | West Reef | Georgia | Tbilisi...,20240430-5543-151220515edd,-1,2311,CONSULT,20240430-5543-151220515edd


In [17]:
len(ce_df['Md5'].unique())
# 这里输出数量对不上,应该是 md5_list = list(group_df['Md5'].unique()) 的问题,存在同一篇新闻对同一个事件抽取了多次

2183402

In [18]:
# stats
results = ce_df.groupby(['Topic']).agg({'timid':['count', 'max','min','mean']})
results.columns = ['n_events', 'nday_max', 'nday_min', 'nday_mean']
results['nday_range'] = results['nday_max'] -results['nday_min'] + 1

In [19]:
results.loc[0:, :].mean()

n_events      414.545562
nday_max      913.516879
nday_min      875.544854
nday_mean     894.322570
nday_range     38.972025
dtype: float64

In [20]:
results.loc[0:, :].max()

n_events      24904.000000
nday_max       2311.000000
nday_min       2294.000000
nday_mean      2305.785714
nday_range      851.000000
dtype: float64

In [21]:
results.loc[0:, :].min()

n_events      21.000000
nday_max       6.000000
nday_min       0.000000
nday_mean      2.643275
nday_range     1.000000
dtype: float64

In [22]:
# split too large complex events by max date range, and max atomic event number

In [23]:
max_n_range = 30
max_n_events = 100

In [24]:
topicid_max = ce_df['Topic'].max()

In [25]:
topicid_max

np.int64(8471)

In [26]:
ce_pd_splitted = []
for topicid in range(0, topicid_max + 1):
    ce_pd = ce_df[ce_df['Topic']==topicid]
    ce_pd = ce_pd.sort_values(by=['day'], ignore_index=True)
    curr_rowid = 0
    while curr_rowid <= (len(ce_pd)-1):
        next_rowid = min(curr_rowid + max_n_events-1, len(ce_pd)-1)
        next_row = ce_pd.iloc[[next_rowid]]
        next_timid = next_row['timid'].values[0]
        curr_row = ce_pd.iloc[[curr_rowid]]
        curr_timid = curr_row['timid'].values[0]
        n_range = next_timid - curr_timid + 1
        if n_range <= max_n_range: # split by max_n_events
            ce_pd_splitted.append(ce_pd[curr_rowid: next_rowid+1])
            curr_rowid = next_rowid+1
        else: # split by n_range
            next_rowid = len(ce_pd[ce_pd['timid'] <= (curr_timid+max_n_range-1)]) - 1
            ce_pd_splitted.append(ce_pd[curr_rowid: next_rowid+1])
            curr_rowid = next_rowid+1    

In [27]:
new_ces = []
for idx, new_ce in enumerate(ce_pd_splitted):
    new_ce = new_ce.rename(columns={'Topic': 'ce_id'})
    new_ce['ce_id'] = [idx] * len(new_ce)
    new_ces.append(new_ce)

In [28]:
len(new_ces)

40475

In [29]:
new_ces[0]

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,ce_id,timid,EventType,Md5_list
0,20201130-5370-4fd072591982_REQUEST_assist,Abdullah II of Jordan,REQUEST,assist,Jordan,2020-11-30,,Jordan,Jordan,,Abdel Fattah Al - Sissi | Abdullah II,,Middle East | Hashemite Kingdom of Jordan,20201130-5370-4fd072591982,0,1064,REQUEST_assist,20201130-5370-4fd072591982
1,20201201-9042-c3ed8e65e9be_COOPERATE,Jordanian Armed Forces,COOPERATE,,governmental institutions,2020-12-01,natural_resource,Jordan,None; None,,Al - Haniti | Yusef Ahmad Al - Huneiti,the Jordanian Armed Forces | the Al - Bashayer...,,20201201-9042-c3ed8e65e9be,0,1065,COOPERATE,20201201-9042-c3ed8e65e9be
2,20201201-9042-c3ed8e65e9be_COOPERATE,Jordanian Armed Forces,COOPERATE,,private institutions,2020-12-01,natural_resource,Jordan,None; None,,Al - Haniti | Yusef Ahmad Al - Huneiti,the Jordanian Armed Forces | the Al - Bashayer...,,20201201-9042-c3ed8e65e9be,0,1065,COOPERATE,20201201-9042-c3ed8e65e9be
3,20201201-9042-c3ed8e65e9be_RETREAT_disarm,Jordanian Armed Forces,RETREAT,disarm,Amman,2020-12-01,natural_resource,Jordan,Jordan,,Al - Haniti | Yusef Ahmad Al - Huneiti,the Jordanian Armed Forces | the Al - Bashayer...,,20201201-9042-c3ed8e65e9be,0,1065,RETREAT_disarm,20201201-9042-c3ed8e65e9be
4,20201207-5378-0c0eec66ce46_AID,European Union,AID,,Jordan,2020-12-07,technology,European Union,Jordan,,Ziad Al - Damour | Imad Shana'a | Corine Andre,the Ministry of Planning | the European Partne...,Jordan River,20201207-5378-0c0eec66ce46,0,1071,AID,20201207-5378-0c0eec66ce46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,20201214-2098-6352e9360f03_ACCUSE_allege,other Gulf countries,ACCUSE,allege,Egypt,2020-12-14,economic | inequality | natural_resource | ter...,None; Saudi Arabia,Egypt; Syria,EGY,,G20 | The Gulf Arab States Cooperation Council,Arab Republic of Egypt | Kingdom of Saudi Arab...,20201214-2098-6352e9360f03,0,1078,ACCUSE_allege,20201214-2098-6352e9360f03
96,20201214-2098-6352e9360f03_ACCUSE_allege,other Gulf countries,ACCUSE,allege,Syria,2020-12-14,economic | inequality | natural_resource | ter...,None; Saudi Arabia,Egypt; Syria,EGY,,G20 | The Gulf Arab States Cooperation Council,Arab Republic of Egypt | Kingdom of Saudi Arab...,20201214-2098-6352e9360f03,0,1078,ACCUSE_allege,20201214-2098-6352e9360f03
97,20201214-2098-6352e9360f03_ACCUSE_allege,Saudi Arabia,ACCUSE,allege,Egypt,2020-12-14,economic | inequality | natural_resource | ter...,None; Saudi Arabia,Egypt; Syria,EGY,,G20 | The Gulf Arab States Cooperation Council,Arab Republic of Egypt | Kingdom of Saudi Arab...,20201214-2098-6352e9360f03,0,1078,ACCUSE_allege,20201214-2098-6352e9360f03
98,20201214-2754-cdfab70b3cc3_REQUEST_meet,Sidi Muhammad Khaled Al - Naciri,REQUEST,meet,Abdullah II of Jordan,2020-12-14,territory,,Jordan,JOR,Sidi Muhammad Khaled Al - Naciri | Abdullah II,The United Nations | the UN Security Council,Meseta Marocaine | Amman | Laayoune | Dead Sea...,20201214-2754-cdfab70b3cc3,0,1078,REQUEST_meet,20201214-2754-cdfab70b3cc3


In [30]:
outliers = ce_df[ce_df['Topic']==-1].sort_values(by=['day'], ignore_index=True)
outliers = outliers.rename(columns={'Topic': 'ce_id'})

In [31]:
outliers

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,ce_id,timid,EventType,Md5_list
0,20180101-0491-8557544d1585_ASSAULT_abduct,Essam Awad,ASSAULT,abduct,Al-Azhar Al-Sharif,2018-01-01,,Egypt,Egypt,,Barakat | Suleiman Al - Atifi | Essam Barakat ...,the House of Representatives,,20180101-0491-8557544d1585,-1,0,ASSAULT_abduct,20180101-0491-8557544d1585
1,20180101-3968-c90ff6322bf4_ACCUSE,Kim Jong-un,ACCUSE,,US,2018-01-01,,North Korea,United States,,Kim Jong - un | Kim,Yonhap,,20180101-3968-c90ff6322bf4,-1,0,ACCUSE,20180101-3968-c90ff6322bf4
2,20180101-3968-c90ff6322bf4_REQUEST,Kim Jong-un,REQUEST,,US,2018-01-01,,North Korea,United States,,Kim Jong - un | Kim,Yonhap,,20180101-3968-c90ff6322bf4,-1,0,REQUEST,20180101-3968-c90ff6322bf4
3,20180101-3609-0c0362e23192_ASSAULT_beat,police,ASSAULT,beat,members of the Hindu organisations,2018-01-01,health,,Syria,,,the Hindu Mahasabha,Mina Bazar | Manhattan | BÅ«ndi | Tiger Hill |...,20180101-3609-0c0362e23192,-1,0,ASSAULT_beat,20180101-3609-0c0362e23192
4,20180101-3609-0c0362e23192_SANCTION,police,SANCTION,,members of the Hindu organisations,2018-01-01,health,,Syria,IND,,the Hindu Mahasabha,Mina Bazar | Manhattan | BÅ«ndi | Tiger Hill |...,20180101-3609-0c0362e23192,-1,0,SANCTION,20180101-3609-0c0362e23192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5472938,20240430-7830-7ce655d47e49_THREATEN,International Criminal Court,THREATEN,,Benjamin Netanyahu,2024-04-30,repression | legal,,Israel,NLD,Herzi Halevi | Netanyahu | Benjamin Netanyahu ...,The International Criminal Court | the Israel ...,The Hague | State of Israel | Kingdom of the N...,20240430-7830-7ce655d47e49,-1,2311,THREATEN,20240430-7830-7ce655d47e49
5472939,20240430-0719-dcc8c49e2861_ASSAULT_destroy,Zionism,ASSAULT,destroy,Zionism,2024-04-30,,Israel,Israel,,,,Boston | Al JumhÅ«rÄ« | al-Aqsa | Satan ÃayÄ±...,20240430-0719-dcc8c49e2861,-1,2311,ASSAULT_destroy,20240430-0719-dcc8c49e2861
5472940,20240430-8742-45a2b038d762_CONSULT,Tamim bin Hamad Al Thani,CONSULT,,Abdel Fattah el-Sisi,2024-04-30,human_security | terrorism | territory,Qatar,None; Egypt,,Abdel - Fattah al - Sisi | Tamim Bin Hamad Al ...,the Egyptian Presidency | Hamas,Gaza | State of Israel | Arab Republic of Egyp...,20240430-8742-45a2b038d762,-1,2311,CONSULT,20240430-8742-45a2b038d762
5472941,20240430-2624-79a5652068b6_CONSULT,Petr Fiala,CONSULT,,Frank-Walter Steinmeier,2024-04-30,technology,Czechia; Germany,Czechia; Germany,CZE,Fiala Fiala | Pavel | Petr Pavel | Frank - Wal...,EU | the European Union | ODS,PraÅ¾skÃ½ hrad | Ukraine | Western GhÄts | Re...,20240430-2624-79a5652068b6,-1,2311,CONSULT,20240430-2624-79a5652068b6


In [32]:
len(outliers["Md5"].unique())

1368971

In [33]:
new_ces.append(outliers)
new_ces_df = pd.concat(new_ces, ignore_index=True)

In [34]:
new_ces_df = new_ces_df.sort_values(by=['day'], ignore_index=True)

In [35]:
new_ces_df

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,ce_id,timid,EventType,Md5_list
0,20180101-5517-9da43a80760f_ACCUSE_investigate,Russian monitors,ACCUSE,investigate,Syria,2018-01-01,intelligence | territory,Russia,Syria,,,Center for Reconciliation of the | the Russian...,Daraa Governorate | ChÃ¢naÃ¯ | Idlib | Syrian ...,20180101-5517-9da43a80760f,-1,0,ACCUSE_investigate,20180101-5517-9da43a80760f
1,20180101-5149-1aaee24ddbaa_REJECT,Jordanian demonstrators,REJECT,,Trump,2018-01-01,,Jordan,,,Pence | Zomlot | Trump | Mike Pence | Mahmoud ...,The United Nations General Assembly,United States | State of Israel | Palestine | ...,20180101-5149-1aaee24ddbaa,-1,0,REJECT,20180101-5149-1aaee24ddbaa
2,20180101-4061-2fdcbda58940_CONCEDE,mourners,CONCEDE,,U.S.-led troops,2018-01-01,religion_ethnicity | gender,,Afghanistan; United States,,,the Associated Press | the Islamic State | Tal...,,20180101-4061-2fdcbda58940,-1,0,CONCEDE,20180101-4061-2fdcbda58940
3,20180101-4061-2fdcbda58940_ACCUSE_disapprove,government,ACCUSE,disapprove,Afghan,2018-01-01,religion_ethnicity | gender,,Afghanistan; United States,,,the Associated Press | the Islamic State | Tal...,,20180101-4061-2fdcbda58940,-1,0,ACCUSE_disapprove,20180101-4061-2fdcbda58940
4,20180101-4061-2fdcbda58940_ACCUSE_investigate,government,ACCUSE,investigate,Afghan,2018-01-01,religion_ethnicity | gender,,Afghanistan; United States,,,the Associated Press | the Islamic State | Tal...,,20180101-4061-2fdcbda58940,-1,0,ACCUSE_investigate,20180101-4061-2fdcbda58940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8984968,20240430-7448-458eb0ff9aa7_CONSULT,Argentine Foreign Minister Diana Mondino,CONSULT,,Han Zheng,2024-04-30,economic,Argentina; None,China,CHN,Mondino | Javier Gonzalez - Olaechea Franco | ...,,Republic of Peru | Sultanate of Oman | Peopleâ...,20240430-7448-458eb0ff9aa7,-1,2311,CONSULT,20240430-7448-458eb0ff9aa7
8984969,20240430-7860-ea1ab78f6e4e_ASSAULT,judge AdÃ¡n QS,ASSAULT,,TC TelevisiÃ³n,2024-04-30,legal,None; None; None; None; Jordan; None; None; No...,Ecuador,ECU,Miguel BP | Jordan TP | Jonathan MP | ' Negro ...,The State Attorney General 's Office | Los Tig...,Guayaquil,20240430-7860-ea1ab78f6e4e,-1,2311,ASSAULT,20240430-7860-ea1ab78f6e4e
8984970,20240430-7860-ea1ab78f6e4e_ASSAULT,judge Jonathan MP,ASSAULT,,TC TelevisiÃ³n,2024-04-30,legal,None; None; None; None; Jordan; None; None; No...,Ecuador,ECU,Miguel BP | Jordan TP | Jonathan MP | ' Negro ...,The State Attorney General 's Office | Los Tig...,Guayaquil,20240430-7860-ea1ab78f6e4e,-1,2311,ASSAULT,20240430-7860-ea1ab78f6e4e
8984971,20240430-0139-3e39df67c7f0_ACCUSE_disapprove,Manasseh Sogavare,ACCUSE,disapprove,western allies,2024-04-30,environment | diplomatic,Solomon Islands,United States; None,,Jeremiah Manele | Sogavare | Manele,Sogavare,Pacific Place | United States | Peopleâs Rep...,20240430-0139-3e39df67c7f0,-1,2311,ACCUSE_disapprove,20240430-0139-3e39df67c7f0


In [36]:
new_ces_df['ce_id'].max()
# 注意outline是-1，所以这里的最大ce_id要加2才是复杂事件数（with outline）

np.int64(40474)

In [37]:
# stats
new_results = new_ces_df.groupby(['ce_id']).agg({'timid':['count', 'max','min','mean']})
new_results.columns = ['n_events', 'nday_max', 'nday_min', 'nday_mean']
new_results['nday_range'] = new_results['nday_max'] -new_results['nday_min'] + 1

In [38]:
new_results

,n_events,nday_max,nday_min,nday_mean,nday_range
ce_id,,,,,
-1,5472943,2311,0,873.421062,2312
0,100,1078,1064,1075.460000,15
1,100,1081,1078,1079.620000,4
2,100,1084,1081,1082.470000,4
3,100,1091,1084,1087.170000,8
...,...,...,...,...,...
40470,50,437,408,421.960000,30
40471,33,451,441,447.636364,11
40472,35,2279,2256,2267.542857,24


In [39]:
new_results.loc[0:, :].mean()

n_events       86.770352
nday_max      883.794145
nday_min      876.212650
nday_mean     880.093416
nday_range      8.581495
dtype: float64

In [40]:
new_results.loc[0:, :].max()

n_events       100.000000
nday_max      2311.000000
nday_min      2310.000000
nday_mean     2310.761905
nday_range      30.000000
dtype: float64

In [41]:
new_results.loc[0:, :].min()

n_events      1.0
nday_max      0.0
nday_min      0.0
nday_mean     0.0
nday_range    1.0
dtype: float64

In [42]:
# filter by min range and min size

In [43]:
len(new_ces)

40476

In [44]:
unfiltered_ces = new_ces[:-1]

In [45]:
len(unfiltered_ces)
# new_ces.append(outliers) 把outline剔除了

40475

In [46]:
min_n_range = 3
min_n_events = 20

In [47]:
invalid_ces = []
valid_ces = []
num_num = 0
num_tim = 0
for ce in unfiltered_ces:
    if len(ce) < min_n_events:
        invalid_ces.append(ce)
        num_num = num_num + 1
    elif len(ce['timid'].unique()) < min_n_range:
        invalid_ces.append(ce)
        num_tim = num_tim + 1
    else:
        valid_ces.append(ce)

In [48]:
len(invalid_ces)

12790

In [49]:
num_num
# 上面拆分到最后的,就有可能小于10

2191

In [50]:
num_tim

10599

In [51]:
new_outliers = pd.concat([outliers]+invalid_ces, ignore_index=True)
new_outliers['ce_id'] = [-1] * len(new_outliers)

In [52]:
len(new_outliers["Md5"].unique())

1575496

In [53]:
new_outliers

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,ce_id,timid,EventType,Md5_list
0,20180101-0491-8557544d1585_ASSAULT_abduct,Essam Awad,ASSAULT,abduct,Al-Azhar Al-Sharif,2018-01-01,,Egypt,Egypt,,Barakat | Suleiman Al - Atifi | Essam Barakat ...,the House of Representatives,,20180101-0491-8557544d1585,-1,0,ASSAULT_abduct,20180101-0491-8557544d1585
1,20180101-3968-c90ff6322bf4_ACCUSE,Kim Jong-un,ACCUSE,,US,2018-01-01,,North Korea,United States,,Kim Jong - un | Kim,Yonhap,,20180101-3968-c90ff6322bf4,-1,0,ACCUSE,20180101-3968-c90ff6322bf4
2,20180101-3968-c90ff6322bf4_REQUEST,Kim Jong-un,REQUEST,,US,2018-01-01,,North Korea,United States,,Kim Jong - un | Kim,Yonhap,,20180101-3968-c90ff6322bf4,-1,0,REQUEST,20180101-3968-c90ff6322bf4
3,20180101-3609-0c0362e23192_ASSAULT_beat,police,ASSAULT,beat,members of the Hindu organisations,2018-01-01,health,,Syria,,,the Hindu Mahasabha,Mina Bazar | Manhattan | BÅ«ndi | Tiger Hill |...,20180101-3609-0c0362e23192,-1,0,ASSAULT_beat,20180101-3609-0c0362e23192
4,20180101-3609-0c0362e23192_SANCTION,police,SANCTION,,members of the Hindu organisations,2018-01-01,health,,Syria,IND,,the Hindu Mahasabha,Mina Bazar | Manhattan | BÅ«ndi | Tiger Hill |...,20180101-3609-0c0362e23192,-1,0,SANCTION,20180101-3609-0c0362e23192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6522388,20240411-5800-fa53494b7509_THREATEN,Ankara,THREATEN,,Kurdistan Workers' Party,2024-04-11,territory,Turkey,,IRQ,Recep Tayyip Erdogan,the Kurdistan Workers ' Party | PKK | Hurriyet...,Ankara | Baghdad | Erbil | SulaymÄnÄ«yah | Re...,20240411-5800-fa53494b7509,-1,2292,THREATEN,20240411-5800-fa53494b7509
6522389,20240422-8295-6e1e78ae1340_ASSAULT,Recep Tayyip ErdoÄan,ASSAULT,,Kurdistan Workers' Party,2024-04-22,diplomatic | territory,Turkey,,IRQ,Recep Tayyip Erdogan | Erdogan | Bassem al - A...,EU | PKK,Republic of Iraq | Deir ez-Zor Governorate | R...,20240422-8295-6e1e78ae1340,-1,2303,ASSAULT,20240422-8295-6e1e78ae1340
6522390,20240422-1869-6e1e78ae1340_ASSAULT,Ankara,ASSAULT,,Kurdistan Workers' Party,2024-04-22,diplomatic | territory,Turkey,,IRQ,Recep Tayyip Erdogan | Erdogan | Mohammed Shia...,EU | PKK,Republic of Iraq | Deir ez-Zor Governorate | R...,20240422-1869-6e1e78ae1340,-1,2303,ASSAULT,20240422-1869-6e1e78ae1340
6522391,20240422-7390-64ac3f843d0f_REQUEST,Recep Tayyip ErdoÄan,REQUEST,,Baghdad,2024-04-22,,Turkey,Iraq,IRQ,Recep Tayyip Erdogan | Erdogan | Bassem al - A...,the Kurdistan Workers â Party | PKK,Republic of Iraq | Republic of Turkey | Baghdad,20240422-7390-64ac3f843d0f,-1,2303,REQUEST,20240422-7390-64ac3f843d0f


In [54]:
new_outliers_grouplist = list(new_outliers.groupby(['Actor1Name', 'EventType', 'Actor2Name', 'day']))
print(len(new_outliers_grouplist))
new_outliers_dedu_list = []
for groupid, group in tqdm(enumerate(new_outliers_grouplist), total=len(new_outliers_grouplist)):
    group_df = group[1]
    md5_list = []
    md5_lists = list(group_df['Md5_list'].unique())
    for md5_list_str in md5_lists:
        md5_list += md5_list_str.split(', ')
    kept_outlier = group_df.head(1).copy()
    kept_outlier['Md5_list'] = ', '.join(md5_list)
    new_outliers_dedu_list.append(kept_outlier.copy())
new_outliers_dedu = pd.concat(new_outliers_dedu_list).sort_values(by=['day'], ignore_index=True)

6522393


100%|██████████| 6522393/6522393 [49:05<00:00, 2214.23it/s]   


In [55]:
# 最开始的过滤考虑了topic,所以这次把新的无效事件加入之后,需要再次过滤
new_outliers_dedu

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,ce_id,timid,EventType,Md5_list
0,20180101-6691-883a38231f77_ACCUSE_investigate,Bashar al-Assad,ACCUSE,investigate,pro government gunmen,2018-01-01,intelligence | military,Syria,None; None,SYR,Fahd Jassem al - Freij | Mohammed Mazen Yousse...,SANA | al - Qaida,Damascus Countryside | Idlib | Damascus | Syri...,20180101-6691-883a38231f77,-1,0,ACCUSE_investigate,20180101-6691-883a38231f77
1,20180101-2577-257024cc458f_SANCTION,Benjamin Netanyahu,SANCTION,,Ayatollahs,2018-01-01,repression | diplomatic,Israel,,,Ayoub Kara | Yisrael Katz | Benjamin Netanyahu...,the Jerusalem Post | Voice of Israel | Yesh At...,ÄªrÄn,20180101-2577-257024cc458f,-1,0,SANCTION,20180101-2577-257024cc458f
2,20180101-6122-e83ef65b242c_RETREAT,Russian tankers,RETREAT,,Four Months Later...,2018-01-01,human_rights | human_rights | technology | tec...,Russia,,,,Reuters | U.N. | the UN Security Council,Pyongyang | Russian Federation | Democratic Pe...,20180101-6122-e83ef65b242c,-1,0,RETREAT,20180101-6122-e83ef65b242c
3,20180101-1810-4d5e62cb2f71_THREATEN_territory,Shia Islam,THREATEN,territory,Israel,2018-01-01,diplomatic | migration,,Israel,,Lapid | Donald Trump | Obama | Trump | Benjami...,the Democratic Party | the Democratic party,United States | State of Israel | Tel Aviv Dis...,20180101-1810-4d5e62cb2f71,-1,0,THREATEN_territory,20180101-1810-4d5e62cb2f71
4,20180101-4585-9f5963ee3de6_REQUEST_assist,China,REQUEST,assist,Russia,2018-01-01,,China; Russia,China; Russia,,Le Maire | Donald Trump | Emmanuel Macron | Br...,Finance | The Wall Street Journal | America Fi...,,20180101-4585-9f5963ee3de6,-1,0,REQUEST_assist,20180101-4585-9f5963ee3de6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6522388,20240430-2762-af276df1eed8_CONSULT,South Korea,CONSULT,,Minister,2024-04-30,technology,South Korea; Australia,Australia; None,AUS,Richard Marles | Shin Won - sik,Yonhap | Hanwha Aerospace | Defense | Hanwha,Commonwealth of Australia | Republic of Korea ...,20240430-2762-af276df1eed8,-1,2311,CONSULT,20240430-2762-af276df1eed8
6522389,20240430-9496-8038294f422e_REQUEST_yield,Hama,REQUEST,yield,Antony Blinken,2024-04-30,,Syria,United States,ISR,Blinken | Antony Blinken,Hamas | Al - Qahera News,State of Israel | Gaza Strip | Middle East | A...,20240430-9496-8038294f422e,-1,2311,REQUEST_yield,20240430-9496-8038294f422e
6522390,20240430-3517-68c0e514bed6_REQUEST_meet,Hama,REQUEST,meet,Ministry Hamas,2024-04-30,territory,Palestinian Territories; Syria,None; None,CHN,Awad | Shahin,the Jerusalem Research Center | Al Jazeera | t...,Palestine | Peopleâs Republic of China | Bei...,20240430-3517-68c0e514bed6,-1,2311,REQUEST_meet,20240430-3517-68c0e514bed6
6522391,20240430-7412-19ef0a909656_CONSULT,Antony Blinken,CONSULT,,Riyadh,2024-04-30,,United States,Saudi Arabia,,Anthony Blinken | Ayman Safadi,State,Gaza Strip | United States | Rafaá¸© | West Ba...,20240430-7412-19ef0a909656,-1,2311,CONSULT,20240430-7412-19ef0a909656


In [56]:
new_outliers_dedu[new_outliers_dedu['timid']==0]

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,ce_id,timid,EventType,Md5_list
0,20180101-6691-883a38231f77_ACCUSE_investigate,Bashar al-Assad,ACCUSE,investigate,pro government gunmen,2018-01-01,intelligence | military,Syria,None; None,SYR,Fahd Jassem al - Freij | Mohammed Mazen Yousse...,SANA | al - Qaida,Damascus Countryside | Idlib | Damascus | Syri...,20180101-6691-883a38231f77,-1,0,ACCUSE_investigate,20180101-6691-883a38231f77
1,20180101-2577-257024cc458f_SANCTION,Benjamin Netanyahu,SANCTION,,Ayatollahs,2018-01-01,repression | diplomatic,Israel,,,Ayoub Kara | Yisrael Katz | Benjamin Netanyahu...,the Jerusalem Post | Voice of Israel | Yesh At...,ÄªrÄn,20180101-2577-257024cc458f,-1,0,SANCTION,20180101-2577-257024cc458f
2,20180101-6122-e83ef65b242c_RETREAT,Russian tankers,RETREAT,,Four Months Later...,2018-01-01,human_rights | human_rights | technology | tec...,Russia,,,,Reuters | U.N. | the UN Security Council,Pyongyang | Russian Federation | Democratic Pe...,20180101-6122-e83ef65b242c,-1,0,RETREAT,20180101-6122-e83ef65b242c
3,20180101-1810-4d5e62cb2f71_THREATEN_territory,Shia Islam,THREATEN,territory,Israel,2018-01-01,diplomatic | migration,,Israel,,Lapid | Donald Trump | Obama | Trump | Benjami...,the Democratic Party | the Democratic party,United States | State of Israel | Tel Aviv Dis...,20180101-1810-4d5e62cb2f71,-1,0,THREATEN_territory,20180101-1810-4d5e62cb2f71
4,20180101-4585-9f5963ee3de6_REQUEST_assist,China,REQUEST,assist,Russia,2018-01-01,,China; Russia,China; Russia,,Le Maire | Donald Trump | Emmanuel Macron | Br...,Finance | The Wall Street Journal | America Fi...,,20180101-4585-9f5963ee3de6,-1,0,REQUEST_assist,20180101-4585-9f5963ee3de6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2277,20180101-5132-80e2b6c6d9d6_COOPERATE,Ahmed Al Mulla,COOPERATE,,Professor Ibrahim Ahmed Omar,2018-01-01,economic | economic | health | health,Bahrain,,,Ibrahim Ahmed Omar | Hamad bin Isa Al Khalifa ...,The House of Representatives | the House of Re...,,20180101-5132-80e2b6c6d9d6,-1,0,COOPERATE,20180101-5132-80e2b6c6d9d6
2278,20180101-0211-d28e3f560736_COERCE,Islamabad,COERCE,,Ajit Doval,2018-01-01,,Pakistan,India,THA,Nasser Khan Janjua | Jadhav | Kulbhushan Jadha...,Dawn | National Security Division,State of Punjab | Islamic Republic of Pakistan...,20180101-0211-d28e3f560736,-1,0,COERCE,20180101-0211-d28e3f560736
2279,20180101-4443-0c19d103a01a_THREATEN,Mahmoud Abbas,THREATEN,,Israel,2018-01-01,territory,Palestinian Territories,Israel,,Mahmud Abbas | Netanyahu,the White House | Likud,United States | State of Israel | West Bank | ...,20180101-4443-0c19d103a01a,-1,0,THREATEN,20180101-4443-0c19d103a01a
2280,20180101-7075-12e9bf6e543e_AID,Rahul Gandhi,AID,,Jignesh Mevani,2018-01-01,legislative | inequality | election,India,None; India; India; India; None,,anti - BJP | Rahul Gandhi | Alpesh Thakor | Ji...,Congress | BJP | the Lok Sabha | The Bharatiya...,GujarÄt | State of GujarÄt,20180101-7075-12e9bf6e543e,-1,0,AID,20180101-7075-12e9bf6e543e


In [57]:
len(new_outliers_dedu['Md5'].unique())

1575496

In [58]:
for idx, valid_ce in enumerate(valid_ces):
    valid_ce['ce_id'] = [idx] * len(valid_ce)

In [59]:
len(valid_ces)

27685

In [60]:
filtered_ces = valid_ces + [new_outliers_dedu]
filtered_ces_df = pd.concat(filtered_ces, ignore_index=True)

In [61]:
filtered_ces_df = filtered_ces_df.sort_values(by=['day'], ignore_index=True)

In [62]:
filtered_ces_df

,Event ID,Actor1Name,Event Type,Event Mode,Actor2Name,day,Contexts,Actor Country,Recipient Country,Country,Story People,Story Organizations,Story Locations,Md5,ce_id,timid,EventType,Md5_list
0,20180101-1820-9cc76a923067_AGREE,Benjamin Netanyahu,AGREE,,Hassan Rouhani,2018-01-01,,Israel,Iran,ISR,Hassan Rouhani | Rouhani | Benjamin Netanyahu ...,,State of Israel | Persian Gulf | Jerusalem | I...,20180101-1820-9cc76a923067,-1,0,AGREE,20180101-1820-9cc76a923067
1,20180101-2956-d39aac46b2e4_AGREE,Pakistan Foreign Office,AGREE,,Donald Trump,2018-01-01,,Pakistan,United States,,Mullah Akhtar Mansoor | Hale | Donald Trump | ...,The US Embassy | Taliban | the Foreign Office ...,United States | AfghÄnistÄn | Al BÄkistÄn ...,20180101-2956-d39aac46b2e4,-1,0,AGREE,20180101-2956-d39aac46b2e4
2,20180101-9293-61015709eb33_SUPPORT,Vasundhara Raje,SUPPORT,,Jaswant Singh Yadav,2018-01-01,,None; India,India; India,,Jaswant Yadav 's | Vasundhara Raje | Karan Yad...,Congress | BJP,Alwar,20180101-9293-61015709eb33,-1,0,SUPPORT,20180101-9293-61015709eb33
3,20180101-3325-27f38c291f54_ACCUSE_allege,Yacoub Sarraf,ACCUSE,allege,Mr. Badri Daher,2018-01-01,military | military,Lebanon; None; None; None; None,None; None; None,,Yacoub Riyad Al - Sarraf | Joseph Aoun | Tony ...,Customs | Army,,20180101-3325-27f38c291f54,-1,0,ACCUSE_allege,20180101-3325-27f38c291f54
4,20180101-1117-dcec4ac6dd72_ACCUSE,The Friends of Zion Museum,ACCUSE,,head of state,2018-01-01,,Israel,,ISR,Mike Evans | Donald Trump | Evans | Mike Pence...,the United Nations | the Friends of Zion Insti...,United States | State of Israel | Republic of ...,20180101-1117-dcec4ac6dd72,-1,0,ACCUSE,20180101-1117-dcec4ac6dd72
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8984968,20240430-7170-251eda3bc887_THREATEN,Benjamin Netanyahu,THREATEN,,Gaza Strip,2024-04-30,,Israel,,PSE,Netanyahu | Benjamin Netanyahu | Antony Blinken,Hamas | State,Gaza | State of Israel | Middle East | Rafaá¸©...,20240430-7170-251eda3bc887,-1,2311,THREATEN,20240430-7170-251eda3bc887
8984969,20240430-3853-a40ce9caa415_PROTEST,Harsimrat Kaur Badal,PROTEST,,Bharatiya Janata Party,2024-04-30,,India,India,,Harsimrat Kaur Badal | Amarinder Singh | Harsi...,AAP | Aam Aadmi Party | BJP | NDA | SAD | Shir...,Bathinda | BudhlÄda | State of Punjab | Mansa,20240430-3853-a40ce9caa415,-1,2311,PROTEST,20240430-3853-a40ce9caa415
8984970,20240430-3517-68c0e514bed6_REQUEST_meet,Fatah,REQUEST,meet,Ministry Hamas,2024-04-30,territory,Palestinian Territories; Syria,None; None,CHN,Awad | Shahin,the Jerusalem Research Center | Al Jazeera | t...,Palestine | Peopleâs Republic of China | Bei...,20240430-3517-68c0e514bed6,-1,2311,REQUEST_meet,20240430-3517-68c0e514bed6
8984971,20240430-3577-d1ec4b9a0103_ACCUSE_investigate,Antony Blinken,ACCUSE,investigate,Hama,2024-04-30,diplomatic | rights_freedoms,United States,Israel; Syria,SAU,Blinken | Antony Blinken,UN Security Council | US State Department | Re...,Gaza | State of Israel | Middle East | Riyadh ...,20240430-3577-d1ec4b9a0103,-1,2311,ACCUSE_investigate,20240430-3577-d1ec4b9a0103


In [63]:
len(filtered_ces_df['Md5'].unique())

2183402

In [64]:
len(filtered_ces_df[filtered_ces_df['ce_id']!=-1]['Md5'].unique())

624692

In [65]:
len(filtered_ces_df[filtered_ces_df['ce_id']!=-1])

2462580

In [66]:
# stats
filtered_results = filtered_ces_df.groupby(['ce_id']).agg({'timid':['count', 'max','min','mean']})
filtered_results.columns = ['n_events', 'nday_max', 'nday_min', 'nday_mean']
filtered_results['nday_range'] = filtered_results['nday_max'] -filtered_results['nday_min'] + 1

In [67]:
filtered_results

,n_events,nday_max,nday_min,nday_mean,nday_range
ce_id,,,,,
-1,6522393,2311,0,859.466368,2312
0,100,1078,1064,1075.460000,15
1,100,1081,1078,1079.620000,4
2,100,1084,1081,1082.470000,4
3,100,1091,1084,1087.170000,8
...,...,...,...,...,...
27680,29,431,403,417.517241,29
27681,50,437,408,421.960000,30
27682,33,451,441,447.636364,11


In [68]:
filtered_results.loc[0:, :].mean()

n_events       88.949973
nday_max      914.127723
nday_min      903.705725
nday_mean     909.064989
nday_range     11.421997
dtype: float64

In [69]:
filtered_results.loc[0:, :].max()

n_events       100.00
nday_max      2311.00
nday_min      2305.00
nday_mean     2308.19
nday_range      30.00
dtype: float64

In [70]:
filtered_results.loc[0:, :].min()

n_events      20.00
nday_max       2.00
nday_min       0.00
nday_mean      0.64
nday_range     3.00
dtype: float64

In [ ]:
filtered_final = filtered_ces_df.rename(columns={'Actor1Name': 'Subject', 'Actor2Name': 'Object', 'EventType': 'Relation', 'day': 'Date'})
filtered_final = filtered_final.loc[:, ["Event ID", "Date", 'Subject', 'Relation', 'Object', "Contexts", "Actor Country", "Recipient Country", "Country", "ce_id", "Md5"]]
filtered_final.to_csv(path_or_buf='./data_mideast_max100_min20/MidEast_Raw.csv', sep=',', index=False)